# Data Collection with the SpaceX REST API

Collects Falcon 9 launch data from `https://api.spacexdata.com/v4`, resolves rocket / launchpad / payload / core IDs, filters to Falcon 9 and exports `data/dataset_part_1.csv`.

> **Note:** this notebook needs internet access. Run it locally or in Colab; outputs are not stored here.

In [ ]:
import requests, pandas as pd, numpy as np
pd.set_option("display.max_columns",None)
BASE="https://api.spacexdata.com/v4"

In [ ]:
# 1. Request launch history
resp=requests.get(f"{BASE}/launches/past"); resp.raise_for_status()
data=pd.json_normalize(resp.json())
data=data[["rocket","payloads","launchpad","cores","flight_number","date_utc"]]
data=data[data["cores"].map(len)==1]
data=data[data["payloads"].map(len)==1]
data["cores"]=data["cores"].map(lambda x:x[0]); data["payloads"]=data["payloads"].map(lambda x:x[0])
data["date"]=pd.to_datetime(data["date_utc"]).dt.date
data=data[data["date"]<=pd.to_datetime("2020-11-13").date()]
data.shape

In [ ]:
# 2. Helper functions that resolve IDs
BoosterVersion=[];PayloadMass=[];Orbit=[];LaunchSite=[];Outcome=[];Flights=[];GridFins=[];Reused=[];Legs=[];LandingPad=[];Block=[];ReusedCount=[];Serial=[];Longitude=[];Latitude=[]
def getBoosterVersion(d):
    for x in d["rocket"]:
        if x: BoosterVersion.append(requests.get(f"{BASE}/rockets/{x}").json()["name"])
def getLaunchSite(d):
    for x in d["launchpad"]:
        if x:
            r=requests.get(f"{BASE}/launchpads/{x}").json()
            Longitude.append(r["longitude"]);Latitude.append(r["latitude"]);LaunchSite.append(r["name"])
def getPayloadData(d):
    for load in d["payloads"]:
        if load:
            r=requests.get(f"{BASE}/payloads/{load}").json()
            PayloadMass.append(r["mass_kg"]);Orbit.append(r["orbit"])
def getCoreData(d):
    for core in d["cores"]:
        if core["core"]:
            r=requests.get(f"{BASE}/cores/{core['core']}").json()
            Block.append(r["block"]);ReusedCount.append(r["reuse_count"]);Serial.append(r["serial"])
        else:
            Block.append(None);ReusedCount.append(None);Serial.append(None)
        Outcome.append(str(core["landing_success"])+" "+str(core["landing_type"]))
        Flights.append(core["flight"]);GridFins.append(core["gridfins"]);Reused.append(core["reused"]);Legs.append(core["legs"]);LandingPad.append(core["landpad"])

In [ ]:
getBoosterVersion(data); getLaunchSite(data); getPayloadData(data); getCoreData(data)

In [ ]:
# 3. Assemble DataFrame, keep Falcon 9, impute payload mass
launch_dict={"FlightNumber":list(data["flight_number"]),"Date":list(data["date"]),"BoosterVersion":BoosterVersion,"PayloadMass":PayloadMass,"Orbit":Orbit,"LaunchSite":LaunchSite,"Outcome":Outcome,"Flights":Flights,"GridFins":GridFins,"Reused":Reused,"Legs":Legs,"LandingPad":LandingPad,"Block":Block,"ReusedCount":ReusedCount,"Serial":Serial,"Longitude":Longitude,"Latitude":Latitude}
df=pd.DataFrame(launch_dict)
data_falcon9=df[df["BoosterVersion"]!="Falcon 1"].copy()
data_falcon9["FlightNumber"]=list(range(1,data_falcon9.shape[0]+1))
data_falcon9["PayloadMass"]=data_falcon9["PayloadMass"].fillna(data_falcon9["PayloadMass"].mean())
data_falcon9.isnull().sum()

In [ ]:
data_falcon9.to_csv("data/dataset_part_1.csv",index=False)
data_falcon9.head()